In [2]:
!pip install openai requests

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   --------------------------- ------------ 0.8/1.1 MB 2.5 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 2.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   -------------------------- ------------- 1.3/2.0 MB 2.3 MB/s eta 0:00:01
   ------------------------------------ --- 1.8/2.0 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.3 MB/s  0:00:00

   ----------------------------------------  0/19 [urllib3]
   ----------------------------------------  0/19 [urllib3]
   ----------------------------------------  0/19 [urllib3]
   -- -------------------------------------  1/19 [typing-extensions]
   ---- ----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from openai import OpenAI
import requests
import json

In [4]:
client = OpenAI(
    api_key="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",
    base_url="https://openrouter.ai/api/v1"
)

In [5]:
def get_order_status(order_id):

    url = f"http://103.86.52.158:8989/get-order?orderNumber={order_id}"

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            data = response.json()
            return json.dumps(data)
        else:
            return f"Backend error: {response.status_code}"

    except Exception as e:
        return f"Request failed: {str(e)}"

In [6]:
print(get_order_status("10100"))

Request failed: HTTPConnectionPool(host='103.86.52.158', port=8989): Max retries exceeded with url: /get-order?orderNumber=10100 (Caused by ConnectTimeoutError(<HTTPConnection(host='103.86.52.158', port=8989) at 0x266ff6db620>, 'Connection to 103.86.52.158 timed out. (connect timeout=10)'))


In [10]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order_status",
            "description": "Get the status of a customer's order using their order number.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "The order number provided by the customer"
                    }
                },
                "required": ["order_id"]
            }
        }
    }
]

In [11]:
messages = [
    {
        "role": "system",
        "content": """
You are a helpful call center agent.

When a user asks about their order status:
1. Ask them for their order number if they haven't provided it.
2. Once they provide the order number, call the get_order_status function.
3. Explain the returned result to the user clearly.
"""
    }
]

In [ ]:
while True:

    user_input = input("User: ")

    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=tools
    )

    print(response)

    message = response.choices[0].message

    if message.tool_calls:

        tool_call = message.tool_calls[0]

        args = json.loads(tool_call.function.arguments)

        order_result = get_order_status(args["order_id"])

        messages.append(message)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": order_result
        })

        final_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages
        )

        reply = final_response.choices[0].message.content

    else:
        reply = message.content

    print("Bot:", reply)

    messages.append({"role": "assistant", "content": reply})

ChatCompletion(id='gen-1775241217-IZXGpbO39Tx1PIILOWPv', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello! How can I assist you today?', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='stop')], created=1775241217, model='openai/gpt-4o-mini', object='chat.completion', service_tier=None, system_fingerprint='fp_e738e3044b', usage=CompletionUsage(completion_tokens=10, prompt_tokens=121, total_tokens=131, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, cache_write_tokens=0, video_tokens=0), cost=2.415e-05, is_byok=False, cost_details={'upstream_inference_cost': 2.415e-05, 'upstream_inference_prompt_cost': 1.815e-05, 'upstream_inference_completions_cost'